In [2]:
pip install pandas sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


In [6]:
# =========================================================
# Cork Pharma MRO Reliability Platform - SAP PM Ready
# STEP 6 - PYTHON OUTPUTS FROM MYSQL STAR TABLES
# SIMULATED LEARNING DATA
#
# Creates exactly 4 CSV files:
# 1. spare_demand_forecast.csv
# 2. asset_reliability.csv
# 3. monthly_energy.csv
# 4. stockout_risk.csv
# =========================================================

import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from getpass import getpass


# =========================================================
# 1. MYSQL CONNECTION
# =========================================================

MYSQL_USER = "root"
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DATABASE = "cork_mro_project"

# Password will NOT appear on screen
MYSQL_PASSWORD = getpass("Enter MySQL password: ")

# Handles passwords containing @, #, %, /, : etc.
encoded_password = quote_plus(MYSQL_PASSWORD)

connection_string = (
    f"mysql+pymysql://{MYSQL_USER}:"
    f"{encoded_password}@"
    f"{MYSQL_HOST}:{MYSQL_PORT}/"
    f"{MYSQL_DATABASE}"
)

engine = create_engine(connection_string)


# =========================================================
# 2. TEST CONNECTION
# =========================================================

try:
    with engine.connect() as connection:
        print("MySQL connection: SUCCESS")
except Exception as e:
    print("MySQL connection: FAILED")
    print(e)
    raise


# =========================================================
# 3. LOAD STAR TABLES
# =========================================================

fact = pd.read_sql(
    "SELECT * FROM fact_breakdown",
    engine
)

dim_asset = pd.read_sql(
    "SELECT * FROM dim_asset",
    engine
)

dim_spare = pd.read_sql(
    "SELECT * FROM dim_spare",
    engine
)

dim_date = pd.read_sql(
    "SELECT * FROM dim_date",
    engine
)


print("\nRows loaded:")
print("fact_breakdown:", len(fact))
print("dim_asset:", len(dim_asset))
print("dim_spare:", len(dim_spare))
print("dim_date:", len(dim_date))


# Check fact table
if len(fact) != 120:
    raise ValueError(
        f"Expected 120 fact_breakdown rows, but found {len(fact)}"
    )

print("\nfact_breakdown check: PASS")


# =========================================================
# 4. JOIN STAR TABLES FOR ANALYSIS
# =========================================================

data = (
    fact
    .merge(
        dim_date[
            [
                "date_key",
                "full_date"
            ]
        ],
        on="date_key",
        how="left"
    )
    .merge(
        dim_asset[
            [
                "asset_key",
                "asset_id",
                "machine_name",
                "equipment_type",
                "location",
                "criticality"
            ]
        ],
        on="asset_key",
        how="left"
    )
    .merge(
        dim_spare[
            [
                "spare_key",
                "spare_id",
                "spare_name",
                "asset_type_link",
                "stock_qty",
                "min_stock",
                "unit_cost_euro",
                "supplier",
                "lead_time_days"
            ]
        ],
        on="spare_key",
        how="left"
    )
)

data["full_date"] = pd.to_datetime(
    data["full_date"]
)

print("Joined analysis rows:", len(data))


# =========================================================
# OUTPUT 1
# 3-MONTH SPARE DEMAND FORECAST
# =========================================================

# Count how many times each spare was used each month

spare_usage = (
    data[
        (data["spare_used"] == "Y")
        & (data["spare_id"].notna())
    ]
    .groupby(
        [
            pd.Grouper(
                key="full_date",
                freq="MS"
            ),
            "spare_id"
        ]
    )
    .size()
    .reset_index(
        name="demand_qty"
    )
)


# Full project period
months = pd.date_range(
    start="2024-09-01",
    end="2025-08-01",
    freq="MS"
)

spare_ids = (
    dim_spare["spare_id"]
    .dropna()
    .unique()
)


# Create every month x every spare
# Missing demand becomes 0

full_index = pd.MultiIndex.from_product(
    [
        months,
        spare_ids
    ],
    names=[
        "full_date",
        "spare_id"
    ]
)

monthly_demand = (
    spare_usage
    .set_index(
        [
            "full_date",
            "spare_id"
        ]
    )
    .reindex(
        full_index,
        fill_value=0
    )
    .reset_index()
)


# Forecast September-November 2025

forecast_rows = []

forecast_months = pd.date_range(
    start="2025-09-01",
    periods=3,
    freq="MS"
)


for spare_id in spare_ids:

    history = (
        monthly_demand[
            monthly_demand["spare_id"]
            == spare_id
        ]
        .sort_values("full_date")
        ["demand_qty"]
        .astype(float)
        .tolist()
    )

    for forecast_month in forecast_months:

        # 3-month moving average
        forecast_value = (
            sum(history[-3:]) / 3
        )

        forecast_rows.append(
            {
                "forecast_month":
                    forecast_month,

                "spare_id":
                    spare_id,

                "forecast_demand":
                    round(
                        forecast_value,
                        2
                    )
            }
        )

        # Rolling forecast:
        # use forecast in next month's average
        history.append(
            forecast_value
        )


forecast = pd.DataFrame(
    forecast_rows
)


# Add spare names

forecast = forecast.merge(
    dim_spare[
        [
            "spare_id",
            "spare_name"
        ]
    ],
    on="spare_id",
    how="left"
)


forecast = forecast[
    [
        "forecast_month",
        "spare_id",
        "spare_name",
        "forecast_demand"
    ]
]


forecast.to_csv(
    "spare_demand_forecast.csv",
    index=False
)

print(
    "\nCreated:"
    " spare_demand_forecast.csv"
)


# =========================================================
# OUTPUT 2
# ASSET RELIABILITY - MTBF / MTTR
# =========================================================

# Assumption:
# 16 hours/day x 300 operating days
# = 4,800 planned hours per asset

PLANNED_HOURS = 4800


reliability = (
    data
    .groupby(
        [
            "asset_id",
            "machine_name",
            "equipment_type",
            "criticality"
        ],
        as_index=False
    )
    .agg(
        failures=(
            "work_order_id",
            "count"
        ),

        total_downtime_hours=(
            "downtime_hours",
            "sum"
        )
    )
)


# Estimated uptime
reliability[
    "estimated_uptime_hours"
] = (
    PLANNED_HOURS
    - reliability[
        "total_downtime_hours"
    ]
)


# MTBF
reliability[
    "MTBF_hours"
] = (
    reliability[
        "estimated_uptime_hours"
    ]
    /
    reliability[
        "failures"
    ]
)


# MTTR
reliability[
    "MTTR_hours"
] = (
    reliability[
        "total_downtime_hours"
    ]
    /
    reliability[
        "failures"
    ]
)


# Round numbers

reliability[
    "total_downtime_hours"
] = reliability[
    "total_downtime_hours"
].round(2)

reliability[
    "estimated_uptime_hours"
] = reliability[
    "estimated_uptime_hours"
].round(2)

reliability[
    "MTBF_hours"
] = reliability[
    "MTBF_hours"
].round(2)

reliability[
    "MTTR_hours"
] = reliability[
    "MTTR_hours"
].round(2)


reliability.to_csv(
    "asset_reliability.csv",
    index=False
)

print(
    "Created:"
    " asset_reliability.csv"
)


# =========================================================
# OUTPUT 3
# MONTHLY ENERGY PER UNIT
# =========================================================

monthly_energy = (
    data
    .set_index(
        "full_date"
    )
    .resample(
        "MS"
    )
    .agg(
        total_energy_kWh=(
            "energy_kWh",
            "sum"
        ),

        total_units_produced=(
            "units_produced",
            "sum"
        )
    )
    .reset_index()
)


monthly_energy[
    "energy_per_unit"
] = (
    monthly_energy[
        "total_energy_kWh"
    ]
    /
    monthly_energy[
        "total_units_produced"
    ]
)


monthly_energy[
    "total_energy_kWh"
] = monthly_energy[
    "total_energy_kWh"
].round(2)


monthly_energy[
    "energy_per_unit"
] = monthly_energy[
    "energy_per_unit"
].round(5)


monthly_energy.to_csv(
    "monthly_energy.csv",
    index=False
)

print(
    "Created:"
    " monthly_energy.csv"
)


# =========================================================
# OUTPUT 4
# STOCKOUT RISK
# =========================================================

# Risk rule:
#
# stock_qty < min_stock
# AND lead_time_days > 7
# AND spare supports a current
# Criticality A asset


critical_a_types = (
    dim_asset[
        (dim_asset["criticality"] == "A")
        &
        (dim_asset["is_current"] == 1)
    ]
    ["equipment_type"]
    .dropna()
    .unique()
)


stockout_risk = dim_spare[
    (
        dim_spare["stock_qty"]
        <
        dim_spare["min_stock"]
    )
    &
    (
        dim_spare["lead_time_days"]
        > 7
    )
    &
    (
        dim_spare[
            "asset_type_link"
        ]
        .isin(
            critical_a_types
        )
    )
].copy()


# Quantity needed to restore
# minimum inventory

stockout_risk[
    "shortage_qty"
] = (
    stockout_risk[
        "min_stock"
    ]
    -
    stockout_risk[
        "stock_qty"
    ]
)


# Cost to restore minimum stock

stockout_risk[
    "reorder_cost_euro"
] = (
    stockout_risk[
        "shortage_qty"
    ]
    *
    stockout_risk[
        "unit_cost_euro"
    ]
).round(2)


stockout_risk = stockout_risk[
    [
        "spare_id",
        "spare_name",
        "asset_type_link",
        "stock_qty",
        "min_stock",
        "shortage_qty",
        "unit_cost_euro",
        "reorder_cost_euro",
        "supplier",
        "lead_time_days"
    ]
]


stockout_risk.to_csv(
    "stockout_risk.csv",
    index=False
)

print(
    "Created:"
    " stockout_risk.csv"
)


# =========================================================
# FINAL STEP 6 CHECK
# =========================================================

print("\n==============================")
print("STEP 6 COMPLETE")
print("==============================")

print(
    "fact_breakdown rows:",
    len(fact)
)

print(
    "Forecast rows:",
    len(forecast)
)

print(
    "Reliability rows:",
    len(reliability)
)

print(
    "Energy months:",
    len(monthly_energy)
)

print(
    "Stockout-risk spares:",
    len(stockout_risk)
)

print("\n4 CSV files saved:")

print(
    "1. spare_demand_forecast.csv"
)

print(
    "2. asset_reliability.csv"
)

print(
    "3. monthly_energy.csv"
)

print(
    "4. stockout_risk.csv"
)

Enter MySQL password:  ········


MySQL connection: SUCCESS

Rows loaded:
fact_breakdown: 120
dim_asset: 11
dim_spare: 10
dim_date: 365

fact_breakdown check: PASS
Joined analysis rows: 120

Created: spare_demand_forecast.csv
Created: asset_reliability.csv
Created: monthly_energy.csv
Created: stockout_risk.csv

STEP 6 COMPLETE
fact_breakdown rows: 120
Forecast rows: 30
Reliability rows: 10
Energy months: 12
Stockout-risk spares: 6

4 CSV files saved:
1. spare_demand_forecast.csv
2. asset_reliability.csv
3. monthly_energy.csv
4. stockout_risk.csv
